# Layer 9 — External Grounding & Validation

**Rolle:** Prüft das Modell gegen externe Realität.
Layer 9 erhebt keine eigenen Felddaten, sondern liest die Outputs von Layer 7 und Layer 8
und vergleicht sie mit externen Quellen und bekannten Ereignissen.

**Architektur:**
```
L7 = Was ist der aktuelle Systemzustand?
L8 = Welche Muster entstehen über Zeit?
L9 = Stimmt das mit externer Realität überein?
```

**Eingaben:**
- `layer7_state.json`
- `layer7_history.jsonl`
- `layer8_state.json`
- Externe APIs (NOAA CPC, NOAA SWPC, NASA EONET)

**Ausgaben:**
- `layer9_state.json` — maschinenlesbares Validierungsergebnis
- `layer9_report.md` — lesbarer Validierungsbericht

## 0. Setup

In [ ]:
import json, os, requests, re
from datetime import datetime, timedelta, timezone
from collections import Counter
import numpy as np

# --- Projektpfade (CWD-unabhaengig, ohne pip install) ---
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / '.project-root').exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))
from atmosphere.paths import layer_state, HISTORY, REPORTS

RUN_TS         = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S.%fZ')
ENGINE_VERSION = '1.0'

L7_STATE   = layer_state(7)
L7_HISTORY = HISTORY
L8_STATE   = layer_state(8)
L9_STATE   = layer_state(9)
L9_REPORT  = REPORTS / 'layer9_report.md'

def evidence_level(n):
    if n is None:   return 'unknown'
    if n < 10:      return 'exploratory_signal'
    if n < 30:      return 'testable'
    if n < 100:     return 'moderate_evidence'
    return 'robust_candidate'

print(f'Layer 9 Engine {ENGINE_VERSION}  |  Run: {RUN_TS}')

## 1. Eingaben laden

In [ ]:
with open(L7_STATE,  encoding='utf-8') as f: l7 = json.load(f)
with open(L8_STATE,  encoding='utf-8') as f: l8 = json.load(f)

l7_history = []
with open(L7_HISTORY, encoding='utf-8') as f:
    for line in f:
        if line.strip(): l7_history.append(json.loads(line))

print(f'Layer 7 state:    {l7["timestamp"]}')
print(f'Layer 8 state:    {l8["timestamp"]}')
print(f'L7 history:       {len(l7_history)} Snapshots')

## 2. Validierungs-Framework

In [ ]:
validation_checks = []

def make_check(id, module, goal, expected, observed, status, details,
               external_source='internal', confidence=None):
    """Standardisiertes Validation-Check-Objekt"""
    return {
        'id':              id,
        'module':          module,
        'goal':            goal,
        'expected':        expected,
        'observed':        observed,
        'status':          status,        # passed / failed / uncertain / inconclusive
        'confidence':      confidence,
        'external_source': external_source,
        'details':         details,
        'timestamp':       RUN_TS,
    }

print('Validierungs-Framework bereit.')

## 3. Modul — ENSO Validation

Vergleicht Layer-2-ENSO-Klassifikation mit NOAA CPC ONI.

In [ ]:
print('── ENSO VALIDATION ──')

enso_module = {'external_source': 'NOAA CPC ONI', 'checks': []}

try:
    r = requests.get('https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt', timeout=15)
    r.raise_for_status()
    oni_vals = []
    for line in r.text.splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0] not in ['SEAS', '']:
            try:
                anom = float(parts[4])
                if -4 <= anom <= 4: oni_vals.append(anom)
            except: continue

    external_oni = oni_vals[-1] if oni_vals else None

    # Modell-ENSO aus L7
    model_enso  = l7.get('enso_context', {})
    model_phase = model_enso.get('phase_class', 'unknown')
    model_oni   = model_enso.get('oni_3month_degC')

    if external_oni is not None:
        if external_oni >= 0.5:    expected_phase = 'el_nino'
        elif external_oni >= 0.2:  expected_phase = 'warm_neutral'
        elif external_oni >= -0.2: expected_phase = 'neutral'
        elif external_oni >= -0.5: expected_phase = 'cool_neutral'
        else:                      expected_phase = 'la_nina'

        match = (model_phase == expected_phase)
        check = make_check(
            id       = 'V_enso_phase',
            module   = 'enso_validation',
            goal     = 'Layer-2-ENSO-Klassifikation stimmt mit offiziellem NOAA ONI überein',
            expected = expected_phase,
            observed = model_phase,
            status   = 'passed' if match else 'failed',
            details  = f'NOAA ONI={external_oni:+.2f}°C → erwartet {expected_phase}. Modell={model_phase} (ONI={model_oni}).',
            external_source = 'NOAA CPC ONI',
            confidence = 0.95,
        )
        validation_checks.append(check)
        enso_module['checks'].append(check['id'])
        enso_module['external_oni'] = external_oni
        enso_module['model_phase']  = model_phase
        print(f'  Erwartet: {expected_phase}  |  Modell: {model_phase}  →  {check["status"]}')
    else:
        print('  ⚠️  Kein externer ONI verfügbar')
        enso_module['external_oni'] = None

except Exception as e:
    print(f'  ❌ ENSO-Validierung fehlgeschlagen: {e}')
    enso_module['error'] = str(e)
    enso_module['external_oni'] = None

## 4. Modul — Space Weather Validation

Vergleicht L0/L4 mit NOAA SWPC Kp + Sturm-Events.

In [ ]:
print('── SPACE WEATHER VALIDATION ──')

sw_module = {'external_source': 'NOAA SWPC', 'checks': []}

try:
    r = requests.get('https://services.swpc.noaa.gov/json/planetary_k_index_1m.json', timeout=10)
    r.raise_for_status()
    kp_data   = r.json()
    recent_kp = [float(e['kp_index']) for e in kp_data[-60:]
                 if e.get('kp_index') is not None]
    external_kp_max = max(recent_kp) if recent_kp else None

    # Externe Kp-Werte
    ext_kp_current = recent_kp[-1]  if recent_kp else None
    ext_kp_max_60m = max(recent_kp) if recent_kp else None

    # Kp aus L7 — None-safe (0.0 ist gültiger Wert, kein or-Fallback!)
    kp_ctx = l7.get('kp_context', {})

    model_kp_1m = kp_ctx.get('Kp_current_1m')
    if model_kp_1m is None:
        model_kp_1m = (
            l7.get('layers', {})
              .get('L4_ionosphere', {})
              .get('key_metrics', {})
              .get('Kp')
        )

    model_kp_60m  = kp_ctx.get('Kp_max_60min')

    model_kp_used = kp_ctx.get('Kp_used_for_score')
    if model_kp_used is None:
        model_kp_used = model_kp_1m

    kp_method  = kp_ctx.get('Kp_score_method', 'unknown')

    l4         = l7.get('layers', {}).get('L4_ionosphere', {})
    storm_flag = l4.get('flags', {}).get('geomagnetic_storm', False)

    kp_validation_context = {
        'model_kp_current_1m': model_kp_1m,
        'model_kp_max_60min':  model_kp_60m,
        'model_kp_used':       model_kp_used,
        'model_kp_method':     kp_method,
        'noaa_kp_current':     ext_kp_current,
        'noaa_kp_max_60min':   ext_kp_max_60m,
    }

    # Zeitfenster wählen — gleiches Fenster bevorzugen
    if model_kp_60m is not None and ext_kp_max_60m is not None:
        kp_delta  = abs(model_kp_60m - ext_kp_max_60m)
        compare_a = f'model_max_60min={model_kp_60m}'
        compare_b = f'noaa_max_60min={ext_kp_max_60m}'
        window    = '60min_max'
    elif model_kp_1m is not None and ext_kp_current is not None:
        kp_delta  = abs(float(model_kp_1m) - ext_kp_current)
        compare_a = f'model_current={model_kp_1m}'
        compare_b = f'noaa_current={ext_kp_current}'
        window    = 'current_1m'
    elif model_kp_used is not None and ext_kp_max_60m is not None:
        kp_delta  = abs(float(model_kp_used) - ext_kp_max_60m)
        compare_a = f'model_used({kp_method})={model_kp_used}'
        compare_b = f'noaa_max_60min={ext_kp_max_60m}'
        window    = 'mixed_window'
    else:
        kp_delta = None
        window   = 'no_data'

    if kp_delta is not None:
        # Status-Logik: <= 1.0 = passed, < 2.0 = uncertain, >= 2.0 = failed
        if window == 'mixed_window':
            kp_status = 'uncertain'
            interp    = 'time_window_mismatch: model and validation use different Kp windows'
        elif kp_delta <= 1.0:
            kp_status = 'passed'
            interp    = f'Kp-Werte konsistent innerhalb Toleranz ({window})'
        elif kp_delta < 2.0:
            kp_status = 'uncertain'
            interp    = f'Kp-Differenz moderat — prüfen ob Zeitfenster/Update-Lag unterschiedlich ({window})'
        else:
            kp_status = 'failed'
            interp    = f'Kp-Differenz groß — möglicher Modellfehler oder starkes Ereignis ({window})'

        kp_validation_context['delta']          = round(kp_delta, 2)
        kp_validation_context['window_used']    = window
        kp_validation_context['interpretation'] = interp
        kp_validation_context['action'] = (
            'Speichere Kp_current_1m, Kp_max_60min, Kp_3h_official getrennt in Layer 0/4'
            if window == 'mixed_window' else 'OK'
        )

        check = make_check(
            id       = 'V_kp_consistency',
            module   = 'space_weather_validation',
            goal     = 'Modell-Kp und NOAA-Kp im gleichen Zeitfenster konsistent',
            expected = f'|Δ Kp| <= 1.0  (Fenster: {window})',
            observed = f'{compare_a}  vs  {compare_b}  →  |Δ|={kp_delta:.2f}',
            status   = kp_status,
            details  = interp,
            external_source = 'NOAA SWPC',
            confidence = 0.90 if window != 'mixed_window' else 0.50,
        )
        sw_module['kp_validation_context'] = kp_validation_context
        validation_checks.append(check)
        sw_module['checks'].append(check['id'])
        print(f'  {compare_a}  vs  {compare_b}')
        print(f'  Fenster: {window}  |Δ|={kp_delta:.2f}  →  {kp_status}')
        if window == 'mixed_window':
            print(f'  ⚠️  {interp}')

    # Sturm-Konsistenz: Kp >= 5 → storm_flag muss True sein
    if external_kp_max is not None and external_kp_max >= 5.0:
        check = make_check(
            id       = 'V_storm_flag',
            module   = 'space_weather_validation',
            goal     = 'Bei Kp >= 5 ist geomagnetic_storm-Flag gesetzt',
            expected = True,
            observed = bool(storm_flag),
            status   = 'passed' if storm_flag else 'failed',
            details  = f'NOAA Kp={external_kp_max:.2f} → Sturm. Modell-Flag={storm_flag}',
            external_source = 'NOAA SWPC',
            confidence = 0.95,
        )
        validation_checks.append(check)
        sw_module['checks'].append(check['id'])
        print(f'  Sturm-Flag bei Kp>=5: {check["status"]}')

except Exception as e:
    print(f'  ❌ Space-Weather-Validierung fehlgeschlagen: {e}')
    sw_module['error'] = str(e)

## 5. Modul — Storm / Atmosphere Validation

Vergleicht L3 mit NASA EONET (laufende Sturm-Events).

In [ ]:
print('── STORM VALIDATION ──')

storm_module = {'external_source': 'NASA EONET', 'checks': []}

try:
    r = requests.get(
        'https://eonet.gsfc.nasa.gov/api/v3/events?status=open&category=severeStorms',
        timeout=15
    )
    r.raise_for_status()
    eonet       = r.json()
    open_storms = len(eonet.get('events', []))

    l3                = l7.get('layers', {}).get('L3_atmosphere', {})
    l3_score          = l3.get('score', 0) or 0
    active_storms_flag = l3.get('flags', {}).get('active_thunderstorms', False)

    if open_storms >= 5:
        expected_min_l3 = 0.30
        match = l3_score >= expected_min_l3
        check = make_check(
            id       = 'V_storm_atmosphere',
            module   = 'storm_validation',
            goal     = f'Bei >= 5 offenen Sturm-Events weltweit ist L3 >= {expected_min_l3}',
            expected = f'L3 >= {expected_min_l3}',
            observed = f'L3 = {l3_score:.3f}',
            status   = 'passed' if match else 'uncertain',
            details  = f'EONET open storms={open_storms}. L3={l3_score:.3f}, active_thunderstorms={active_storms_flag}',
            external_source = 'NASA EONET',
            confidence = 0.60,
        )
        validation_checks.append(check)
        storm_module['checks'].append(check['id'])
        storm_module['open_storms'] = open_storms
        print(f'  EONET open storms={open_storms}  |  L3={l3_score:.3f}  →  {check["status"]}')
    else:
        print(f'  EONET open storms={open_storms} — zu wenig für Vergleich (< 5)')
        storm_module['open_storms'] = open_storms

except Exception as e:
    print(f'  ❌ Storm-Validierung fehlgeschlagen: {e}')
    storm_module['error'] = str(e)

## 6. Modul — Schumann Validation

Aktuell kein zuverlässiger öffentlicher Echtzeit-Endpunkt verfügbar.
Layer 6 bleibt `model_expected_not_observed` bis eine Datenquelle gefunden wird.

In [ ]:
print('── SCHUMANN VALIDATION ──')

schumann_module = {
    'external_source': 'no_public_realtime_feed',
    'status':          'pending_data_source',
    'note':            'Echtzeit-Schumann-Daten sind nicht öffentlich verfügbar. '
                       'Layer 6 bleibt model_expected_not_observed.',
    'checks': [],
}
check = make_check(
    id       = 'V_schumann_data_availability',
    module   = 'schumann_validation',
    goal     = 'Externe Schumann-Resonanz-Messdaten für Vergleich verfügbar',
    expected = 'real-time SR1 amplitude/frequency feed',
    observed = 'no public feed available',
    status   = 'inconclusive',
    details  = 'Layer 6 läuft als modelliert, nicht gemessen. Validierung blockiert.',
    external_source = 'none',
    confidence = 0.0,
)
validation_checks.append(check)
schumann_module['checks'].append(check['id'])
print(f'  {check["status"]} — {check["details"]}')

## 7. Modul — Internal Consistency Validation

Prüft logische Beziehungen zwischen Layern.

In [ ]:
print('── INTERNAL CONSISTENCY ──')

consistency_module = {'external_source': 'internal_rules', 'checks': []}

# Regel 1: confirmed_shift → downstream_score > 0.4
cg = l7.get('cavity_gate', {})
if cg.get('type') == 'confirmed_shift':
    ds = l7.get('meta_scores', {}).get('downstream_score', 0)
    check = make_check(
        id       = 'V_consistency_cavity_downstream',
        module   = 'consistency_validation',
        goal     = 'confirmed_shift impliziert downstream_score > 0.4',
        expected = 'downstream > 0.4',
        observed = f'downstream = {ds:.3f}',
        status   = 'passed' if ds > 0.4 else 'failed',
        details  = f'cavity_gate.type={cg.get("type")}, downstream={ds:.3f}',
    )
    validation_checks.append(check)
    consistency_module['checks'].append(check['id'])
    print(f'  cavity confirmed → downstream > 0.4: {check["status"]}')

# Regel 2: anomalous_resonance_state → L3, L5, L6 alle > 0.35
if l7.get('system_state') == 'anomalous_resonance_state':
    ms = l7.get('meta_scores', {})
    all_high = (
        ms.get('activation_score', 0) > 0.35 and
        ms.get('electric_score', 0)   > 0.35 and
        ms.get('resonance_score', 0)  > 0.35
    )
    check = make_check(
        id       = 'V_consistency_anomalous',
        module   = 'consistency_validation',
        goal     = 'anomalous_resonance_state impliziert L3+L5+L6 alle > 0.35',
        expected = 'all > 0.35',
        observed = f'L3={ms.get("activation_score",0):.3f}, L5={ms.get("electric_score",0):.3f}, L6={ms.get("resonance_score",0):.3f}',
        status   = 'passed' if all_high else 'failed',
        details  = 'Meta-Score-Konsistenz für anomalous_resonance_state',
    )
    validation_checks.append(check)
    consistency_module['checks'].append(check['id'])
    print(f'  anomalous → alle downstream > 0.35: {check["status"]}')

# Regel 3: baseline_tags ∩ signal_tags = ∅
baseline = set(l7.get('baseline_tags', []))
signal   = set(l7.get('signal_tags',   []))
overlap  = baseline & signal
check = make_check(
    id       = 'V_consistency_tags',
    module   = 'consistency_validation',
    goal     = 'Baseline-Tags und Signal-Tags sind disjunkt',
    expected = 'no overlap',
    observed = f'overlap = {list(overlap) if overlap else "[]"}',
    status   = 'passed' if not overlap else 'failed',
    details  = f'baseline_n={len(baseline)}, signal_n={len(signal)}',
)
validation_checks.append(check)
consistency_module['checks'].append(check['id'])
print(f'  Tag-Trennung baseline ∩ signal = ∅: {check["status"]}')

# Regel 4: seasonal_transition_state → preparation_score > 0.45 und downstream < 0.35
if l7.get('system_state') == 'seasonal_transition_state':
    ms   = l7.get('meta_scores', {})
    prep = ms.get('preparation_score', 0)
    ds   = ms.get('downstream_score', 0)
    ok   = prep > 0.45 and ds < 0.35
    check = make_check(
        id       = 'V_consistency_seasonal',
        module   = 'consistency_validation',
        goal     = 'seasonal_transition_state: preparation > 0.45 und downstream < 0.35',
        expected = 'prep > 0.45 AND downstream < 0.35',
        observed = f'prep={prep:.3f}, downstream={ds:.3f}',
        status   = 'passed' if ok else 'uncertain',
        details  = 'Meta-Score-Konsistenz für seasonal_transition_state',
    )
    validation_checks.append(check)
    consistency_module['checks'].append(check['id'])
    print(f'  seasonal → prep > 0.45 + downstream < 0.35: {check["status"]}')

## 8. Modul — Known-Event Backtest

Sucht in der History nach bekannten Mustern und prüft sie gegen Erwartungen.

In [ ]:
print('── KNOWN-EVENT BACKTEST ──')

backtest_module = {'external_source': 'l7_history', 'checks': []}

# Backtest 1: anomalous_resonance_state nur abends?
anomal_snaps = [s for s in l7_history if s.get('system_state') == 'anomalous_resonance_state']
if anomal_snaps:
    evenings = sum(
        1 for s in anomal_snaps
        if datetime.fromisoformat(s['timestamp'].replace('Z','')).hour + 2 >= 18
    )
    pct_evening = round(evenings / len(anomal_snaps) * 100, 1)
    check = make_check(
        id       = 'V_backtest_carnegie_anomalous',
        module   = 'known_event_backtest',
        goal     = 'anomalous_resonance_state tritt überwiegend (>= 80%) abends auf',
        expected = '>= 80% evening',
        observed = f'{pct_evening}% evening ({evenings}/{len(anomal_snaps)})',
        status   = 'passed' if pct_evening >= 80 else 'uncertain' if pct_evening >= 60 else 'failed',
        details  = f'Carnegie-Tagesgang-Backtest aus History. n={len(anomal_snaps)}.',
        confidence = 0.85 if len(anomal_snaps) >= 5 else 0.5,
    )
    validation_checks.append(check)
    backtest_module['checks'].append(check['id'])
    print(f'  Anomalous abends: {pct_evening}%  →  {check["status"]}  (n={len(anomal_snaps)})')
else:
    print('  Keine anomalous_resonance_state in History — Backtest übersprungen')

# Backtest 2: ENSO-Konsistenz über History
ext_oni = enso_module.get('external_oni')
if ext_oni is not None and ext_oni >= 0.2:
    warm_snaps = [
        s for s in l7_history
        if (s.get('enso_context') or {}).get('phase_class')
        in ('warm_neutral', 'el_nino_candidate', 'el_nino')
    ]
    pct_warm = round(len(warm_snaps) / len(l7_history) * 100, 1) if l7_history else 0
    check = make_check(
        id       = 'V_backtest_enso_consistency',
        module   = 'known_event_backtest',
        goal     = 'Bei externem ONI >= 0.2 sind >= 50% der Snapshots warm-klassifiziert',
        expected = '>= 50% warm',
        observed = f'{pct_warm}% warm',
        status   = 'passed' if pct_warm >= 50 else 'uncertain' if pct_warm >= 30 else 'failed',
        details  = f'ONI={ext_oni:+.2f}, warm_snaps={len(warm_snaps)}/{len(l7_history)}',
        confidence = 0.85,
    )
    validation_checks.append(check)
    backtest_module['checks'].append(check['id'])
    print(f'  ENSO-Konsistenz History: {pct_warm}% warm  →  {check["status"]}')

# Backtest 3: L2-Score im Mittel hoeher bei warm_neutral/el_nino als bei neutral?
l2_by_enso = {}
for s in l7_history:
    phase = (s.get('enso_context') or {}).get('phase_class', 'unknown')
    l2_v  = s.get('layers', {}).get('L2_surface_zone', {}).get('score')
    if l2_v is not None:
        l2_by_enso.setdefault(phase, []).append(l2_v)

l2_warm_mean    = np.mean(l2_by_enso.get('warm_neutral', [0])) if l2_by_enso.get('warm_neutral') else None
l2_neutral_mean = np.mean(l2_by_enso.get('neutral', [0]))      if l2_by_enso.get('neutral') else None

if l2_warm_mean is not None and l2_neutral_mean is not None:
    check = make_check(
        id       = 'V_backtest_l2_enso',
        module   = 'known_event_backtest',
        goal     = 'L2 ist im Mittel höher bei warm_neutral als bei neutral',
        expected = 'L2_warm > L2_neutral',
        observed = f'L2_warm={l2_warm_mean:.3f}, L2_neutral={l2_neutral_mean:.3f}',
        status   = 'passed' if l2_warm_mean > l2_neutral_mean else 'uncertain',
        details  = f'n_warm={len(l2_by_enso.get("warm_neutral",[]))}, n_neutral={len(l2_by_enso.get("neutral",[]))}',
        confidence = 0.70,
    )
    validation_checks.append(check)
    backtest_module['checks'].append(check['id'])
    print(f'  L2 warm_neutral vs neutral: {l2_warm_mean:.3f} vs {l2_neutral_mean:.3f}  →  {check["status"]}')

## 9. Aggregat — Validation Score

In [ ]:
status_counts = Counter(c['status'] for c in validation_checks)
n_total       = len(validation_checks)
n_passed      = status_counts.get('passed', 0)
n_failed      = status_counts.get('failed', 0)
n_uncertain   = status_counts.get('uncertain', 0)
n_inconcl     = status_counts.get('inconclusive', 0)

# Score: passed=1, uncertain=0.5, failed=0, inconclusive ignoriert
testable = n_total - n_inconcl
validation_score = round(
    (n_passed + 0.5 * n_uncertain) / testable, 4
) if testable > 0 else None

failed_checks    = [c for c in validation_checks if c['status'] == 'failed']
uncertain_checks = [c for c in validation_checks if c['status'] == 'uncertain']

# Modell-Anpassungs-Vorschläge
adjustments = []
for c in failed_checks:
    if c['module'] == 'enso_validation':
        adjustments.append(f'Layer 2: ENSO-Klassifikations-Schwellen überprüfen ({c["id"]} failed)')
    elif c['module'] == 'space_weather_validation':
        adjustments.append(f'Layer 0/4: Kp-Refresh-Logik prüfen ({c["id"]} failed)')
    elif c['module'] == 'consistency_validation':
        adjustments.append(f'Layer 7: State-Klassifikations-Regeln prüfen ({c["id"]} failed)')
    elif c['module'] == 'known_event_backtest':
        adjustments.append(f'Layer 8: Hypothese {c["id"]} braucht mehr Daten oder neue Schwellen')

# L8-Hypothesen gegen Checks mappen — Promotion-Lock respektieren.
# Eine confound-blockierte Hypothese (zirkulaer/proxy) darf NICHT den Anschein
# externer Validierung bekommen, auch wenn ein Check zufaellig matcht.
validated_hypotheses = []
for h in l8.get('hypothesis_candidates', []):
    related = [c for c in validation_checks
               if h['id'].lower() in c['id'].lower()
               or any(k in c['details'].lower()
                      for k in [h.get('source_metric','').lower()] if k)]
    if not related:
        continue
    indep      = h.get('evidence_independence', 'unclassified')
    promotable = h.get('include_in_model_logic', h.get('promotion_eligible', True))
    confound_blocked = (indep in ('confounded_circular', 'confounded_proxy')
                        or promotable is False)
    validated_hypotheses.append({
        'hypothesis_id':         h['id'],
        'evidence_independence': indep,
        'confound_blocked':      confound_blocked,
        'validation_status':     'confound_blocked' if confound_blocked else 'validated',
        'check_results':         [{'id': c['id'], 'status': c['status']} for c in related],
    })

print('=' * 78)
print('VALIDATION SUMMARY')
print('=' * 78)
print(f'  Total checks:      {n_total}')
print(f'  ✅ Passed:         {n_passed}')
print(f'  ❌ Failed:         {n_failed}')
print(f'  ⚠️  Uncertain:     {n_uncertain}')
print(f'  ❓ Inconclusive:   {n_inconcl}')
print(f'  Validation Score:  {validation_score}')
print(f'  Evidence Level:    {evidence_level(n_total)}')
print()
if adjustments:
    print('  Model Adjustment Suggestions:')
    for a in adjustments:
        print(f'    → {a}')

## 10. Export — layer9_state.json + layer9_report.md

In [ ]:
layer9_state = {
    'timestamp':         RUN_TS,
    'engine_version':    ENGINE_VERSION,
    'layer':             9,
    'name':              'External Grounding & Validation',
    'validation_scope':  'early_model_validation',
    'input_sources': [
        'layer7_state',
        'layer7_history',
        'layer8_state',
        'external_apis',
    ],
    'validation_modules': {
        'enso_validation':          enso_module,
        'space_weather_validation': sw_module,
        'storm_validation':         storm_module,
        'schumann_validation':      schumann_module,
        'consistency_validation':   consistency_module,
        'known_event_backtest':     backtest_module,
    },
    'validation_checks':  validation_checks,
    'aggregate': {
        'n_total':          n_total,
        'n_passed':         n_passed,
        'n_failed':         n_failed,
        'n_uncertain':      n_uncertain,
        'n_inconclusive':   n_inconcl,
        'validation_score': validation_score,
        'evidence_level':   evidence_level(n_total),
    },
    'failed_checks':                failed_checks,
    'uncertain_checks':             uncertain_checks,
    'validated_hypotheses':         validated_hypotheses,
    'model_adjustment_suggestions': adjustments,
}

# numpy-Bereinigung
def _to_python(obj):
    if isinstance(obj, dict):        return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):        return [_to_python(v) for v in obj]
    if isinstance(obj, np.bool_):    return bool(obj)
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return None if np.isnan(obj) else float(obj)
    return obj

layer9_state = _to_python(layer9_state)

with open(L9_STATE, 'w', encoding='utf-8') as f:
    json.dump(layer9_state, f, indent=2, ensure_ascii=False)
print(f'✅ {L9_STATE} gespeichert')

# Markdown-Report
lines = [
    '# Layer 9 — External Grounding & Validation',
    '',
    f'**Run:** {RUN_TS}',
    f'**Validation Score:** {validation_score}  ({evidence_level(n_total)})',
    '',
    '## Aggregat',
    f'- ✅ Passed:       {n_passed}',
    f'- ❌ Failed:       {n_failed}',
    f'- ⚠️  Uncertain:  {n_uncertain}',
    f'- ❓ Inconclusive: {n_inconcl}',
    '',
    '## Failed Checks',
]
for c in failed_checks:
    lines.append(f'- **{c["id"]}** ({c["module"]}): {c["details"]}')
if not failed_checks:
    lines.append('- keine')

lines += ['', '## Uncertain Checks']
for c in uncertain_checks:
    lines.append(f'- **{c["id"]}** ({c["module"]}): {c["details"]}')
if not uncertain_checks:
    lines.append('- keine')

lines += ['', '## Model Adjustment Suggestions']
for a in adjustments:
    lines.append(f'- {a}')
if not adjustments:
    lines.append('- keine')

lines += ['', '## Validation Checks (alle)']
for c in validation_checks:
    icon = {'passed':'✅','failed':'❌','uncertain':'⚠️','inconclusive':'❓'}.get(c['status'],'?')
    lines.append(f'- {icon} **{c["id"]}**: {c["goal"]}')
    lines.append(f'  - Erwartet: {c["expected"]}')
    lines.append(f'  - Beobachtet: {c["observed"]}')

with open(L9_REPORT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'✅ {L9_REPORT} gespeichert')